# FTSE 100 Banking Analysis | Public Pilot

A small, standalone demonstration of a **banking research dashboard**. It retrieves public market prices, creates basic metrics, visualises them, and optionally explains the selected view with OpenAI.

**Run:** Execute Sections 1–3 in Google Colab or Jupyter. Section 4 is optional and requires your own API key. Market-data retrieval requires an internet connection.

| Marker | Public pilot mapping | What is demonstrated |
| --- | --- | --- |
| 🟦 S1 | User input | Bank, date window and optional question |
| 🟩 S3 | Public data | Adjusted daily market prices |
| 🟪 S6 | Python analysis | Return and volatility summary |
| 🟪 S7 | AI interpretation | Optional explanation of supplied evidence only |
| 🟦 S9 | Response | Interactive charts, summary and answer |

**Scope:** Selected UK-listed banking shares. This is **not** the full research system, and it does not implement the original agent, news pipeline, advanced quantitative methods or framework validation checks.


## 1. Setup | ⚙️

Install the small set of public dependencies. Re-run after restarting the environment.

In [ ]:
%pip -q install yfinance pandas numpy plotly ipywidgets openai

In [ ]:
import os
import json
from html import escape

import numpy as np
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown, HTML

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

BANKS = {
    "Barclays": "BARC.L",
    "HSBC": "HSBA.L",
    "Lloyds": "LLOY.L",
    "NatWest": "NWG.L",
    "Standard Chartered": "STAN.L",
}
print("Pilot ready. Select a bank and load its data in Section 3.")

## 2. Public data and basic metrics | 🟩 S3 · 🟪 S6

**Does:** Fetch split/dividend-adjusted daily closing prices and compute a period return and annualised volatility. Dates and quoted price units come from Yahoo Finance; prices are not necessarily denominated in GBP.

**Limits:** Data may be delayed or incomplete. The sample is a selected set of banks rather than a verified current index membership list.

In [ ]:
def load_prices(symbol, period="6mo"):
    df = yf.Ticker(symbol).history(period=period, interval="1d", auto_adjust=True, actions=False, timeout=15)
    if df.empty or "Close" not in df:
        raise ValueError("No price history returned. Try another period or bank.")
    df = df[["Close"]].copy()
    df["Close"] = pd.to_numeric(df["Close"], errors="coerce")
    df = df.loc[np.isfinite(df["Close"]) & (df["Close"] > 0)].sort_index()
    df = df.loc[~df.index.duplicated(keep="last")]
    if len(df) < 3:
        raise ValueError("Not enough daily observations for this view.")
    return df


def summarise(df):
    close = df["Close"]
    daily = close.pct_change().dropna()
    return {
        "From": df.index[0].strftime("%Y-%m-%d"),
        "To": df.index[-1].strftime("%Y-%m-%d"),
        "Observations": len(df),
        "Last adjusted close": round(float(close.iloc[-1]), 4),
        "Adjusted-close return (%)": round(float((close.iloc[-1] / close.iloc[0] - 1) * 100), 2),
        "Annualised volatility (%)": round(float(daily.std() * np.sqrt(252) * 100), 2),
    }

## 3. Interactive dashboard | 🟦 S1 · 🟪 S6 · 🟦 S9

Choose the bank and history window, then click **Load prices**. You can change the chart type and moving-average slider without downloading the data again. The **Explain view** button is optional (Section 4).

*Lightweight public interface; the private notebook's agent workflow and validation logic are intentionally not included.*

In [ ]:
bank = widgets.Dropdown(options=list(BANKS), description="Bank:", layout=widgets.Layout(width="360px"))
period = widgets.SelectionSlider(
    options=[("1 month", "1mo"), ("3 months", "3mo"), ("6 months", "6mo"), ("1 year", "1y")],
    value="6mo", description="History:", continuous_update=False,
    layout=widgets.Layout(width="360px"),
)
chart_type = widgets.ToggleButtons(options=["Line", "Area"], description="Chart:")
average = widgets.IntSlider(value=10, min=1, max=40, description="Average:", continuous_update=False)
question = widgets.Textarea(
    value="Summarise this bank's price performance and the limits of the data.",
    description="Ask AI:", layout=widgets.Layout(width="95%", height="70px"),
)
load_button = widgets.Button(description="Load prices", button_style="primary", icon="refresh")
ai_button = widgets.Button(description="Explain view", button_style="info", icon="comment", disabled=True)
status = widgets.HTML("Select a bank and load prices to begin.")
chart_output, answer_output = widgets.Output(), widgets.Output()
state = {"data": None, "summary": None, "ai_connected": False}


def render_view(_=None):
    if state["data"] is None:
        return
    df = state["data"]
    summary = state["summary"]
    close = df["Close"]
    daily = close.pct_change().mul(100)
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.12,
                        row_heights=[0.73, 0.27])
    fig.add_trace(go.Scatter(
        x=df.index, y=close, mode="lines", name="Adjusted close",
        line={"color": "#2563eb", "width": 2.5},
        fill="tozeroy" if chart_type.value == "Area" else None,
        fillcolor="rgba(37,99,235,0.10)"), row=1, col=1)
    if average.value > 1:
        fig.add_trace(go.Scatter(
            x=df.index, y=close.rolling(average.value).mean(), mode="lines",
            name=f"{average.value}-day average", line={"color": "#f59e0b", "width": 2}), row=1, col=1)
    fig.add_trace(go.Bar(
        x=df.index[1:], y=daily.iloc[1:], name="Daily return",
        marker_color=np.where(daily.iloc[1:] >= 0, "#10b981", "#f43f5e")), row=2, col=1)
    fig.update_layout(
        template="plotly_white", height=490, hovermode="x unified",
        title=f"{bank.value} | adjusted close and daily return",
        margin={"l": 55, "r": 20, "t": 70, "b": 35},
        legend={"orientation": "h", "y": 1.12},
        font={"color": "#334155"},
    )
    fig.update_yaxes(title_text="Yahoo quote units", row=1, col=1)
    fig.update_yaxes(title_text="Return (%)", row=2, col=1)
    with chart_output:
        clear_output(wait=True)
        display(HTML(
            '<div style="background:#f8fafc;padding:12px 16px;border-radius:9px;'
            'border:1px solid #e2e8f0;margin-top:8px">'
            f'<b>Period return:</b> {summary["Adjusted-close return (%)"]:+.2f}% &nbsp;&nbsp; '
            f'<b>Annualised volatility:</b> {summary["Annualised volatility (%)"]:.2f}% &nbsp;&nbsp; '
            f'<b>As of:</b> {escape(summary["To"])}'
            '</div>'
        ))
        fig.show()
        display(pd.DataFrame(summary.items(), columns=["Metric", "Value"]))
        display(Markdown("**Source:** Yahoo Finance through `yfinance`, adjusted daily prices. "
                         "Figures describe historical observations, not forecasts."))
    answer_output.clear_output()
    status.value = "Chart ready. Optionally enter a question and click Explain view."


def reset_view(_=None):
    state.update(data=None, summary=None)
    ai_button.disabled = True
    chart_output.clear_output()
    answer_output.clear_output()
    status.value = "Selection changed. Click Load prices."


def load_view(_):
    reset_view()
    status.value = "Retrieving public daily prices..."
    try:
        df = load_prices(BANKS[bank.value], period.value)
        state.update(data=df, summary=summarise(df))
        ai_button.disabled = not state["ai_connected"]
        render_view()
    except Exception as exc:
        status.value = f"Could not load data: {escape(str(exc))}"


bank.observe(reset_view, names="value")
period.observe(reset_view, names="value")
chart_type.observe(render_view, names="value")
average.observe(render_view, names="value")
load_button.on_click(load_view)

display(widgets.VBox([
    widgets.HTML(
      '<div style="background:#0f172a;color:white;padding:22px;border-radius:12px">'
      '<div style="color:#38bdf8;font-size:11px;letter-spacing:2px">BANKING RESEARCH · PUBLIC PILOT</div>'
      '<div style="font-size:24px;font-weight:700;margin:8px 0">Explore the market</div>'
      '<div style="color:#cbd5e1">Public prices · basic metrics · optional AI</div></div>'),
    widgets.HBox([bank, period]), load_button,
    widgets.HBox([chart_type, average]), chart_output,
    question, ai_button, status, answer_output,
], layout=widgets.Layout(width="100%", max_width="1100px", padding="10px", gap="10px")))

## 4. Optional AI explanation | 🟪 S7

To use **Explain view**, set `OPENAI_API_KEY` in the *runtime* (never in a published notebook). Optionally set `OPENAI_MODEL`; the default here is `gpt-4.1-mini`. The dashboard works without an API key. A successful request may incur API charges.

**Run:** Execute the code cell below to attach the button. The model only receives a small summary of the **currently loaded** price data, not the full internal research workflow.

In [ ]:
def explain_view(_):
    if state["data"] is None:
        return
    with answer_output:
        clear_output()
        if not os.getenv("OPENAI_API_KEY"):
            print("Set OPENAI_API_KEY in the runtime before using AI explanations.")
            return
        ai_button.disabled = True
        try:
            from openai import OpenAI
            df = state["data"]
            evidence = {
                "bank": bank.value,
                "source": "Yahoo Finance adjusted daily closing prices",
                "summary": state["summary"],
                "latest_10_closes": [
                    {"date": d.strftime("%Y-%m-%d"), "close": round(float(v), 4)}
                    for d, v in df["Close"].tail(10).items()
                ],
                "question": question.value.strip() or "Explain this view briefly.",
            }
            response = OpenAI().responses.create(
                model=os.getenv("OPENAI_MODEL", "gpt-4.1-mini"),
                store=False,
                instructions=(
                    "Give a brief explanation using only the supplied price evidence. "
                    "Treat returns as adjusted-close returns. Distinguish facts from interpretation. "
                    "Do not invent current quotes, news, reasons for price moves or predictions. "
                    "State the data limitations."
                ),
                input=json.dumps(evidence),
            )
            if not response.output_text:
                raise RuntimeError("The model returned no explanation.")
            display(Markdown(response.output_text))
        except Exception as exc:
            print(f"AI explanation unavailable: {exc}")
        finally:
            ai_button.disabled = False


ai_button.on_click(explain_view)
state["ai_connected"] = True
ai_button.disabled = state["data"] is None
print("Optional AI button connected.")

# To enter a key without saving it to this notebook, run separately when needed:
# from getpass import getpass
# os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")

---

### Pilot boundaries

This public sample deliberately omits the original **multi-tool agent, query/check system, news retrieval, full universe analyses, extended quantitative work and internal prompts**. The OpenAI response is **not independently verified**. It is only an explanation of the displayed price summary and last 10 closing observations. The user must verify claims before relying on them.

Data attribution: Yahoo Finance (accessed through `yfinance`). Historical performance is not a prediction or investment advice.
